# Irina — executable SWE-bench + local Qwen + grouped GRPO

This notebook replaces the previous demonstration-only cells with an actual **one-task, four-rollout** experiment. It uses the code in the published Irina repository, not a simulated model or optimizer. **A working Docker daemon is required** for untrusted agent shell execution; standard hosted Colab commonly lacks one. The notebook deliberately stops rather than running unsafe shell commands on the host. The educational reward is random by design and is **not evidence of patch correctness**. The realistic reward backend remains unverified.

**Execution contract:** run cells in order. Success means you can inspect Qwen-generated tokens, independent trajectories, patches, rewards, gradients, and a LoRA checkpoint. No checkpoint or optimizer success is claimed if the relevant assertion fails. The four original flowcharts are linked below and rendered in the README.


In [ ]:
# 0. User settings — run this first
MODE = "educational"  # "realistic" requires the official SWE-bench evaluator
MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
GROUP_SIZE = 4
MAX_AGENT_TURNS = 4
SEED = 42
TASK_INDEX = 0
REPO_URL = "https://github.com/rderakhshan/SWE.git"
PROJECT_DIR = "/content/Irina"
OUTPUT_DIR = "/content/irina_runs"
assert MODE in ("educational", "realistic")
print({"mode": MODE, "model": MODEL_ID, "group_size": GROUP_SIZE, "turns": MAX_AGENT_TURNS})


In [ ]:
# 1. Bootstrap with uv, targeting the notebook's *current* interpreter
import os, sys, shutil, subprocess, pathlib
project = pathlib.Path(PROJECT_DIR)
if not (project / "pyproject.toml").exists():
    if project.exists() and any(project.iterdir()):
        raise RuntimeError(f"{project} exists but is not the Irina project; inspect it before cloning")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(project)], check=True)
uv = shutil.which("uv") or str(pathlib.Path.home() / ".local/bin/uv")
if not pathlib.Path(uv).exists():
    subprocess.run(["sh", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
    uv = str(pathlib.Path.home() / ".local/bin/uv")
assert pathlib.Path(uv).exists(), "uv executable not found"
extras = "train,realistic" if MODE == "realistic" else "train"
subprocess.run([uv, "pip", "install", "--python", sys.executable, "--system", "-e", f"{project}[{extras}]"], check=True)
import irina
print("Kernel:", sys.executable, "| Irina:", irina.__file__)


## 0. Full pipeline — original diagram


📊 **Flowchart:** [Full pipeline](../README.md#1-swe-bench--odysseus--qwen--grpo--full-pipeline) — GitHub does not render Mermaid inside notebook cells, so the diagram is rendered in the README.


## 1. Preflight: GPU, Docker, and dependencies

**This is a hard gate.** A Colab GPU does not provide a Docker daemon. Without a secure execution backend, do not continue with agent tool calls. Run on a GPU VM with Docker or supply a separately validated sandbox implementation.


In [ ]:
import torch, shutil, subprocess, importlib.util
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
for pkg in ("transformers", "peft", "datasets"):
    assert importlib.util.find_spec(pkg) is not None, f"Missing dependency: {pkg}"
if not shutil.which("docker"):
    raise RuntimeError("Docker executable missing. Hosted Colab usually cannot run this isolated SWE-bench agent pipeline. Use a Docker-enabled GPU VM.")
check = subprocess.run(["docker", "info"], capture_output=True, text=True, timeout=20)
if check.returncode != 0:
    raise RuntimeError("Docker daemon unavailable: " + check.stderr[-1200:])
print("Docker daemon is available; secure execution preflight passed")


## 2. Load the real SWE-bench Verified task


In [ ]:
from irina.rl.tasks import load_task
task = load_task(TASK_INDEX)
print("Instance:", task.instance_id)
print("Repository:", task.repo, "base commit:", task.base_commit)
print("Problem statement:\n", task.problem_statement[:1500])


## 3. Prepare isolated workspaces

The pipeline checks out the same base commit independently for every rollout. This probe validates GitHub access and commit checkout; it does **not** execute tests or mutate the original repository.


In [ ]:
from irina.rl.workspace import checkout, collect_patch, valid_patch
with checkout(task.repo, task.base_commit) as probe:
    print("Checked out:", probe)
    print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=probe, text=True).strip())


## 4. Load Qwen and LoRA — real weights, real model

The first run downloads model weights. It may take minutes and allocate GPU memory. This is not an API call to a commercial provider.


In [ ]:
import random
from irina.providers.qwen import QwenProvider
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
provider = QwenProvider(MODEL_ID)
trainable = sum(p.numel() for p in provider.model.parameters() if p.requires_grad)
assert trainable > 0, "No trainable LoRA parameters"
print("Model loaded; trainable LoRA parameters:", trainable)
print("Model device:", next(provider.model.parameters()).device)


## 5. One agent rollout — original diagram


📊 **Flowchart:** [One agent rollout](../README.md#5-zoom-in--what-happens-during-one-agent-rollout) — GitHub does not render Mermaid inside notebook cells, so the diagram is rendered in the README.


## 6. Execute four independent rollouts and collect patches

This is the expensive section: Qwen generates real tokens; Irina may invoke its real file and sandboxed shell tools. Each rollout begins from the same repository commit. The code writes **actual trajectories and patches**, even when no usable patch is produced. Invalid or empty patches receive zero reward.


In [ ]:
import json, time
from dataclasses import asdict
from irina import Harness, Policy
from irina.sandbox import DockerSandbox
from irina.rl.rewards import synthetic_reward, realistic_reward, RewardResult
run_dir = pathlib.Path(OUTPUT_DIR) / time.strftime("%Y%m%d-%H%M%S")
run_dir.mkdir(parents=True, exist_ok=False)
records, rewards, reward_details, rollout_info = [], [], [], []
for i in range(GROUP_SIZE):
    print(f"\n=== Rollout {i+1}/{GROUP_SIZE} ===", flush=True)
    with checkout(task.repo, task.base_commit) as workdir:
        provider.generations.clear()
        events = []
        agent = Harness(workdir, provider=provider, policy=Policy("yolo"),
                        sandbox=DockerSandbox(), on_event=events.append,
                        session_enabled=False)
        t0 = time.monotonic()
        result = agent.run(task.problem_statement, max_turns=MAX_AGENT_TURNS)
        patch = collect_patch(workdir)
        applicable = valid_patch(patch)
        if not applicable:
            reward = RewardResult(0.0, "invalid_patch", MODE)
        elif MODE == "educational":
            reward = synthetic_reward(patch, SEED, i)
        else:
            reward = realistic_reward(task, patch, f"{run_dir.name}-{i}")
        generation_records = list(provider.generations)
        (run_dir / f"patch-{i}.diff").write_text(patch)
        (run_dir / f"trajectory-{i}.json").write_text(json.dumps({
            "events": events, "messages": result["messages"],
            "reason": result["reason"], "reward": asdict(reward)
        }, indent=2, default=str))
        records.append(generation_records)
        rewards.append(reward.value)
        reward_details.append(asdict(reward))
        info = {"rollout": i, "seconds": round(time.monotonic()-t0, 1),
                "generations": len(generation_records),
                "generated_tokens": sum(len(g["tokens"]) for g in generation_records),
                "patch_bytes": len(patch.encode()), "patch_valid": applicable,
                "reward": reward.value, "reward_status": reward.status}
        rollout_info.append(info)
        print(info, flush=True)
assert len(records) == GROUP_SIZE
print("Artifacts:", run_dir)


## 7. Two reward paths — original diagram


📊 **Flowchart:** [Two reward paths](../README.md#6-zoom-in--two-reward-paths) — GitHub does not render Mermaid inside notebook cells, so the diagram is rendered in the README.


In [ ]:
print("Reward mode:", MODE)
for row in rollout_info: print(row)
if MODE == "educational":
    print("WARNING: rewards are seeded synthetic numbers, NOT correctness scores")
else:
    print("Realistic verifier results; inspect official logs before trusting them")


## 8. Grouped rollouts and GRPO update — original diagram


📊 **Flowchart:** [Grouped rollouts and GRPO update](../README.md#7-zoom-in--grouped-rollouts-and-the-grpo-update) — GitHub does not render Mermaid inside notebook cells, so the diagram is rendered in the README.


## 9. Compute group-relative advantages from the *actual* rewards


In [ ]:
from irina.rl.training import advantages, train_group
assert all(r is not None for r in rewards), "Verifier returned a missing reward"
adv = advantages(rewards)
print("Actual rewards:", rewards)
print("Actual advantages:", adv.tolist())
print("Recorded Qwen generations:", [len(g) for g in records])


## 10. Real generated-token loss and LoRA optimizer step

`train_group` recomputes token log probabilities from the generated tokens, builds a clipped group-relative objective, runs backpropagation and calls `optimizer.step()` when there is a nonzero training signal. It returns `updated=False` for zero-variance rewards or missing generated tokens. The current implementation is a **single-step GRPO-style objective**, not a fully benchmarked reference GRPO trainer.


In [ ]:
# Snapshot a trainable parameter to independently verify that a parameter changed.
name, param = next((n, p) for n, p in provider.model.named_parameters() if p.requires_grad)
before = param.detach().cpu().clone()
metrics = train_group(provider, records, rewards)
after = param.detach().cpu()
changed = not torch.equal(before, after)
print("Training metrics:", metrics)
print("First tracked LoRA tensor:", name, "changed:", changed)
if not metrics.get("updated"):
    print("NO OPTIMIZER UPDATE: inspect reward variance, generation records, and patch validity")


## 11. Save the adapter and experimental evidence


In [ ]:
from irina.rl.config import Config
config = Config(mode=MODE, model_id=MODEL_ID, group_size=GROUP_SIZE,
                max_turns=MAX_AGENT_TURNS, seed=SEED, task_index=TASK_INDEX,
                output_dir=OUTPUT_DIR)
checkpoint = run_dir / "lora_checkpoint"
provider.model.save_pretrained(checkpoint)
provider.tokenizer.save_pretrained(checkpoint)
summary = {"mode": MODE, "model_id": MODEL_ID, "instance_id": task.instance_id,
           "rewards": rewards, "reward_details": reward_details,
           "advantages": adv.tolist(), "rollouts": rollout_info,
           "training": metrics, "tracked_parameter_changed": changed,
           "checkpoint": str(checkpoint)}
(run_dir / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
print("Adapter files:", [p.name for p in checkpoint.iterdir()])


## 12. Interpretation and known limitations

- **No fabricated success:** `updated=False` means no LoRA optimization occurred. A saved adapter alone is not evidence of learning. If all four patches are invalid, all rewards will be zero and the optimizer step is skipped.
- **Docker requirement:** stock hosted Colab usually fails the preflight. This notebook cannot safely provide untrusted SWE-bench shell execution there without an external sandbox.
- **Educational reward:** random, seeded per rollout, and unrelated to whether a patch solves the issue. A nonzero gradient does not mean improved coding ability.
- **Realistic reward:** the project's official SWE-bench harness invocation/report parsing has not been verified end to end and may need adaptation to the installed `swebench` version.
- **Patch validity:** `valid_patch` is a preliminary syntax check; it does not guarantee applicability to the clean base checkout or correctness.
- **Research scope:** the local Qwen/LoRA implementation is experimental, not a reproducible SWE-bench score or a production-quality GRPO implementation. The original Odysseus Python source was not provided.

**Do not interpret fast completion as success.** Inspect the rollout token counts, patches, rewards, `training.updated`, parameter comparison, and saved summary.
